# Uji penerimaan SLR_09 v0.8.2

BELUM DIJALANKAN.50 acuan FINAL082 dan671 token baru masih PENDING, belum mendapat review manusia, dan belum pernah diinferensikan. Review bintang pada FINAL081 tidak mengesahkan set baru ini. Notebook mengimpor Excel reviewer, memvalidasi binding kode/bobot dan cakupan, lalu membekukan acuan sebelum model dimuat. Bobot SENA/NARA tetap dari v0.8.1; pipeline field diperbaiki pada v0.8.2. Hasil memenuhi metrik tetap memerlukan penerimaan mutu.


In [ ]:
from pathlib import Path
import os,sys,json,subprocess
from datetime import datetime,timezone
ROOT=Path(os.environ['SLR09_PACKAGE_ROOT']).resolve()
WORK=Path(os.environ['SLR09_WORK_DIR']).resolve()
REVIEW=Path(os.environ['SLR09_REVIEW_XLSX']).resolve()
SENA=Path(os.environ['SLR09_SENA_ARCHIVE']).resolve()
NARA=Path(os.environ['SLR09_NARA_ARCHIVE']).resolve()
sys.path.insert(0,str(ROOT/'code/acceptance'))
from import_review_xlsx import convert
from evaluate_acceptance import validate,write


In [ ]:
# Penjagaan ini berjalan sebelum model dimuat atau acuan hasil impor disimpan.
bundle=convert(REVIEW,root=ROOT,validate_ready=True)
rows,preflight=validate(bundle,ROOT)
RUN=WORK/('final_acceptance_v082_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
RUN.mkdir(parents=True,exist_ok=False)
reviewed=RUN/'reviewed_reference.json'
write(reviewed,bundle)
print(json.dumps(preflight,ensure_ascii=False,indent=2))


In [ ]:
cmd=[sys.executable,str(ROOT/'code/acceptance/evaluate_acceptance.py'),'--root',str(ROOT),'--review',str(reviewed),'--output',str(RUN/'evidence'),'--sena-zip',str(SENA),'--nara-zip',str(NARA),'--device','cpu','--environment-label','CODEX_CPU_REPRESENTATIVE']
process=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
for line in process.stdout:print(line,end='',flush=True)
process.wait()
if process.returncode:raise RuntimeError('Evaluasi gagal; periksa workflow_result dan error_trace yang disimpan.')
report=json.loads((RUN/'evidence/evaluation.json').read_text())
print(json.dumps({k:report[k] for k in ['checks','metric_targets_met','decision','quality_acceptance_approved','card_done']},ensure_ascii=False,indent=2))


Jika gagal dan hasil dipakai untuk perbaikan, seluruh acuan yang telah diuji menjadi development. Siapkan acuan baru untuk putaran penerimaan berikutnya. Jangan mengubah target atau label setelah melihat prediksi untuk membuat putaran ini lulus.
